In [100]:
import re
import numpy as np
import pandas as pd
import os

Add support for multiples artists at once

In [101]:
list_files = os.listdir("..\Corpus")

In [102]:
global_corpus = []
for i in list_files :
    if "corpus_RNN" in i  :
        with open(f"..\Corpus\{i}", "r", encoding="utf-8", errors="replace") as f:
            corpus = f.read()
#            print("Nb of songs :",len(re.findall(r"<END>", corpus)), "\nNb of featurings :",len(re.findall(r"<BEGINNING>True.*?<END>", corpus, flags=re.DOTALL)))
            global_corpus.extend([corpus])
one_corpus = "".join(global_corpus)

### Featuring will be removed from the Markov Chain as this model will only generate the lyrics of one artist, so the structure of the songs can't be of a featuring.

In [103]:
corpus_solo = re.sub(r"<BEGINNING>True.*?<END>\n\n", " ", one_corpus, flags=re.DOTALL)

## Removing the songs with only one part identified

In [104]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus_solo)]

for i in range(len(all_parts)-3) :
    nn = len(all_parts)-3 -i
    if all_parts[nn-2][0] == "BEGINNING" :
        if all_parts[nn-1][0] == "END_SECTION" :
            if all_parts[nn][0] == "END" :
                part_1_corp = corpus_solo[:all_parts[nn-2][1]]
                part_2_corp = corpus_solo[all_parts[nn][2]:]
                corpus_solo = part_1_corp+part_2_corp

### I will need to regroup each song and then count the number of occurences from each of my parts

In [105]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus_solo)]

count = 0
beg = 0
endi =0

for i in all_parts :
    if i[0] == "BEGINNING" :
        count = 0
        beg = i[1]
    elif i[0] == "END_SECTION" :
        continue
    else : 
        if i[0] == "END" :
            count+=1
            endi = i[2]
            if count >=10 :
                print(count, beg, endi)
        else : 
            count+=1

#corpus_solo = corpus_solo[:beg] + corpus_solo[endi:]

12 5511 7693
11 38268 40552
10 90353 92762
10 256188 259560
10 324538 328771
10 381435 385693
10 395878 400473
10 409344 413421
11 465065 469059
10 723328 727926


Now we will need to annotate the parts by the number of times they appeared.

This will also help in finding errors in the lyrics or structure in the corpus

In [106]:
parts = re.findall(r"<(\w+)>", corpus_solo) #Identify each part
regrouped = " ".join(parts) #re.findall gave back a list, we need a full text to perform next modif
regrouped = re.sub("END_SECTION ", "", regrouped) 
splitted = regrouped.split("END")

splitted = [i.strip() for i in splitted[:-1]]

In [107]:
def add_cumsum_tags(parts):
    counts = {}
    result = []
    for p in parts:
        counts[p] = counts.get(p, 0)
        if counts[p] != 0 :
            result.append(f"{p}_{counts[p]}")
        else : 
            result.append(f"{p}")
        counts[p] += 1
    return result

def transform_to_num_structure(splitted) :
    #This function will serve only if I decide to aggregate more songs from other rapper because we need to have a sufficient nb of sample
    #or we will have a huge nb of single interaction such as couplet_3 refrain_3 ...
    new_splitted = []
    
    for split in splitted :
        interm = add_cumsum_tags(split.split(" "))
        new_splitted.append(interm)

    count_all = {}
        
    for elem in new_splitted :
        for length in range(len(elem)-1) :
            name = elem[length]+" "+elem[length+1]
            interm = count_all.get(name,0)+1
            count_all[name] = interm
    
    return new_splitted, count_all

In [108]:
new_split, count_all = transform_to_num_structure(splitted)

Maybe I will collect data structure of multiple artists and then aggregate to get a better Markov Chain but if I do this, it will be in the future not now

In [109]:
count_all

{'BEGINNING INTRO': 95,
 'INTRO COUPLET': 73,
 'COUPLET REFRAIN': 107,
 'REFRAIN COUPLET_1': 107,
 'COUPLET_1 REFRAIN_1': 102,
 'REFRAIN_1 COUPLET_2': 29,
 'INTRO REFRAIN': 19,
 'REFRAIN COUPLET': 50,
 'COUPLET REFRAIN_1': 48,
 'REFRAIN_1 COUPLET_1': 37,
 'COUPLET_1 REFRAIN_2': 32,
 'REFRAIN_2 PONT': 1,
 'PONT REFRAIN_3': 1,
 'BEGINNING COUPLET': 133,
 'COUPLET_1 PONT': 6,
 'PONT COUPLET_2': 2,
 'COUPLET_2 REFRAIN_1': 3,
 'REFRAIN_1 PONT_1': 5,
 'PONT_1 COUPLET_3': 1,
 'COUPLET_3 REFRAIN_2': 1,
 'REFRAIN_2 COUPLET_4': 1,
 'COUPLET_4 OUTRO': 1,
 'COUPLET COUPLET_1': 38,
 'COUPLET_1 COUPLET_2': 21,
 'COUPLET_2 COUPLET_3': 7,
 'BEGINNING REFRAIN': 38,
 'COUPLET_1 OUTRO': 11,
 'COUPLET_2 PONT': 3,
 'PONT COUPLET_3': 2,
 'COUPLET_3 COUPLET_4': 5,
 'COUPLET PONT': 31,
 'PONT REFRAIN': 11,
 'COUPLET_1 PONT_1': 12,
 'PONT_1 REFRAIN_1': 5,
 'REFRAIN_1 REFRAIN_2': 3,
 'REFRAIN OUTRO': 3,
 'OUTRO INTRO': 2,
 'INTRO COUPLET_1': 1,
 'COUPLET_2 REFRAIN_2': 19,
 'REFRAIN REFRAIN_1': 5,
 'REFRAIN_1 CO

In [110]:
def count_each_interact(splitted) :
    
    for i in range(len(splitted)) :
        splitted[i] = splitted[i]+" END"

    good_parts = (" ".join(splitted)).split(" ")

    dict_glob = {}
    for i in range(len(good_parts)-1) :
        try : 
            dict_glob[f"{good_parts[i]} {good_parts[i+1]}"] += 1
        except :
            dict_glob[f"{good_parts[i]} {good_parts[i+1]}"] = 1

    beginn = {}
    intro = {}
    couplet = {}
    pont = {}
    refrain = {}
    outro = {}

    order = ["BEGINNING","INTRO", "COUPLET", "PONT", "REFRAIN", "OUTRO", "END"]
    list_dicti = [beginn, intro, couplet, pont, refrain, outro]
    for part in list_dicti :
        for j in order :
            part[j] = 0    
    
    for i in dict_glob.keys() :
        actual = i.split(" ")[0]
        next = i.split(" ")[1]

        if actual == "BEGINNING" :  
            beginn[next] = dict_glob[i]

        elif actual == "INTRO" :
            intro[next] = dict_glob[i]

        elif actual == "COUPLET" :
            couplet[next] = dict_glob[i]
        elif actual == "PONT" :
            pont[next] = dict_glob[i]
        elif actual == "REFRAIN" :
            refrain[next] = dict_glob[i]
        elif actual == "OUTRO" :
            outro[next] = dict_glob[i]

    return list_dicti

all_count = count_each_interact(splitted)

In [111]:
all_count = count_each_interact(splitted)

## Some errors can be found in the differents parts, we will remove what we think is impossible

In [112]:
all_count[0] #begin

{'BEGINNING': 0,
 'INTRO': 95,
 'COUPLET': 133,
 'PONT': 0,
 'REFRAIN': 38,
 'OUTRO': 2,
 'END': 0}

In [113]:
all_count[0]["OUTRO"] = 0

In [114]:
all_count[1] #intro

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 75,
 'PONT': 2,
 'REFRAIN': 20,
 'OUTRO': 0,
 'END': 1}

In [115]:
all_count[1]["END"] = 0

In [116]:
all_count[2] #couplet

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 75,
 'PONT': 55,
 'REFRAIN': 319,
 'OUTRO': 41,
 'END': 60}

In [117]:
all_count[3] #pont

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 31,
 'PONT': 0,
 'REFRAIN': 34,
 'OUTRO': 6,
 'END': 0}

In [118]:
all_count[4] #refrain

{'BEGINNING': 0,
 'INTRO': 1,
 'COUPLET': 236,
 'PONT': 14,
 'REFRAIN': 12,
 'OUTRO': 51,
 'END': 109}

In [119]:
all_count[4]["INTRO"] = 0

Intro isn't an error because I found songs where there was two intro and one for each rapper

In [120]:
all_count[5] #outro

{'BEGINNING': 0,
 'INTRO': 2,
 'COUPLET': 0,
 'PONT': 0,
 'REFRAIN': 0,
 'OUTRO': 0,
 'END': 98}

In [121]:
all_count[5]["INTRO"] = 0

## Now we need to obtain the proba for each state to go to another

In [123]:
all_count

[{'BEGINNING': 0,
  'INTRO': 95,
  'COUPLET': 133,
  'PONT': 0,
  'REFRAIN': 38,
  'OUTRO': 0,
  'END': 0},
 {'BEGINNING': 0,
  'INTRO': 0,
  'COUPLET': 75,
  'PONT': 2,
  'REFRAIN': 20,
  'OUTRO': 0,
  'END': 0},
 {'BEGINNING': 0,
  'INTRO': 0,
  'COUPLET': 75,
  'PONT': 55,
  'REFRAIN': 319,
  'OUTRO': 41,
  'END': 60},
 {'BEGINNING': 0,
  'INTRO': 0,
  'COUPLET': 31,
  'PONT': 0,
  'REFRAIN': 34,
  'OUTRO': 6,
  'END': 0},
 {'BEGINNING': 0,
  'INTRO': 0,
  'COUPLET': 236,
  'PONT': 14,
  'REFRAIN': 12,
  'OUTRO': 51,
  'END': 109},
 {'BEGINNING': 0,
  'INTRO': 0,
  'COUPLET': 0,
  'PONT': 0,
  'REFRAIN': 0,
  'OUTRO': 0,
  'END': 98}]

In [122]:
matrix = []
for i in all_count :
    inter_matrix = []
    for j in i :
        inter_matrix.append(i[j]/sum(i.values()))
    matrix.append(inter_matrix)

In [91]:
order = ["<BEGINNING>","<INTRO>", "<COUPLET>", "<PONT>", "<REFRAIN>", "<OUTRO>", "<END>"]

In [92]:
print(order[:])
print(matrix)

['<BEGINNING>', '<INTRO>', '<COUPLET>', '<PONT>', '<REFRAIN>', '<OUTRO>', '<END>']
[[0.0, 0.35714285714285715, 0.5, 0.0, 0.14285714285714285, 0.0, 0.0], [0.0, 0.0, 0.7731958762886598, 0.020618556701030927, 0.20618556701030927, 0.0, 0.0], [0.0, 0.0, 0.13636363636363635, 0.1, 0.58, 0.07454545454545454, 0.10909090909090909], [0.0, 0.0, 0.43661971830985913, 0.0, 0.4788732394366197, 0.08450704225352113, 0.0], [0.0, 0.0, 0.5592417061611374, 0.03317535545023697, 0.02843601895734597, 0.12085308056872038, 0.25829383886255924], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]]


In [93]:
def generate_a_song_structure(matrix) :
    
    song_struct = [0]

    index = [i for i, p in enumerate(matrix[0]) if p>0]
    proba = [p for p in matrix[0] if p > 0]

    cumsum = np.cumsum(proba)
    r = np.random.rand()

    idx = np.searchsorted(cumsum, r)
    selected_value = index[idx] 
    song_struct.append(selected_value)

    end = False

    while end == False :

        index = [i for i, p in enumerate(matrix[selected_value]) if p>0]
        proba = [p for p in matrix[selected_value] if p > 0]

        cumsum = np.cumsum(proba)
        r = np.random.rand()

        idx = np.searchsorted(cumsum, r)
        selected_value = index[idx]
        song_struct.append(selected_value)

        if selected_value == 6 :
            end = True

    print([order[i] for i in song_struct])

In [95]:
for run in range(5) :
    generate_a_song_structure(matrix)

['<BEGINNING>', '<REFRAIN>', '<END>']
['<BEGINNING>', '<COUPLET>', '<COUPLET>', '<REFRAIN>', '<END>']
['<BEGINNING>', '<REFRAIN>', '<COUPLET>', '<COUPLET>', '<PONT>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<END>']
['<BEGINNING>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<OUTRO>', '<END>']
['<BEGINNING>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<OUTRO>', '<END>']


This seems ok for now. 

In [96]:
pd.concat((pd.DataFrame(matrix),pd.DataFrame(order).transpose()))

,0,1,2,3,4,5,6
0,0.0,0.357143,0.5,0.0,0.142857,0.0,0.0
1,0.0,0.0,0.773196,0.020619,0.206186,0.0,0.0
2,0.0,0.0,0.136364,0.1,0.58,0.074545,0.109091
3,0.0,0.0,0.43662,0.0,0.478873,0.084507,0.0
4,0.0,0.0,0.559242,0.033175,0.028436,0.120853,0.258294
5,0.0,0.0,0.0,0.0,0.0,0.0,1.0
0,<BEGINNING>,<INTRO>,<COUPLET>,<PONT>,<REFRAIN>,<OUTRO>,<END>


So the matrix will be exported and used during the model generation of text

In [97]:
pd.concat((pd.DataFrame(matrix),pd.DataFrame(order).transpose())).to_csv("transition_matrix.csv", index=False)

In [98]:
for n in re.finditer(r"<\w+>\n<END_SECTION>", corpus_solo, flags=re.DOTALL) :
    print(n)

<re.Match object; span=(209020, 209041), match='<INTRO>\n<END_SECTION>'>
<re.Match object; span=(210398, 210418), match='<PONT>\n<END_SECTION>'>
<re.Match object; span=(308583, 308604), match='<OUTRO>\n<END_SECTION>'>
<re.Match object; span=(335621, 335641), match='<PONT>\n<END_SECTION>'>
